# Adversarial Debiasing

## What this notebook does:
Trains an adversarially debiased LSTM that is explicitly penalised
for encoding race information in its internal representations.

## Architecture:
- **Main model:** 3-layer BiLSTM (identical to original) predicting mortality
- **Adversary:** small MLP that tries to predict race from the LSTM hidden state
- **Training:** adversarial. LSTM is penalised when adversary succeeds

## Fairness metrics reported:
- AUPRC (primary metric)
- Equal Opportunity (TPR equality across groups)
- Equalized Odds (TPR + FPR equality)
- Balanced Error Rate Equality (BER equality)

## Lambda values tested:
- λ=0.0 no fairness constraint
- λ=1.0 moderate fairness constraint
- λ=3.0 strong fairness constraint
- λ=5.0 very strong fairness constraint

---
## 1. Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
from pathlib import Path
from sklearn.metrics import (
    average_precision_score, roc_auc_score,
    precision_recall_curve, roc_curve
)
from sklearn.model_selection import StratifiedKFold
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset

warnings.filterwarnings('ignore')
pd.set_option('display.float_format', '{:.4f}'.format)

plt.rcParams.update({
    'figure.dpi': 120,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': True,
    'grid.alpha': 0.3,
    'font.size': 11,
})

OUT_DIR  = Path('/home/ino/thesis_outputs')
ADV_DIR  = OUT_DIR / 'adversarial'
ADV_DIR.mkdir(exist_ok=True)

KEEP_RACES   = ['White', 'Black', 'Hispanic', 'Asian']
RACE_TO_INT  = {g: i for i, g in enumerate(KEEP_RACES)}
N_FOLDS      = 5
RANDOM_STATE = 42
N_BOOTSTRAP  = 2000
N_EPOCHS     = 50
BATCH_SIZE   = 512
LR_MAIN      = 5e-4
LR_ADV       = 1e-3

# Lambda values. fairness constraint strength
# 0 = no constraint, higher = stronger fairness pressure
LAMBDAS = [0.0, 1.0, 3.0, 5.0]

# Classification threshold for binary fairness metrics
THRESHOLD = 0.5

LAMBDA_COLORS = {
    0.0: '#7f8c8d',
    1.0: '#3498db',
    3.0: '#e67e22',
    5.0: '#e74c3c',
}

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
n_gpus = torch.cuda.device_count()
print(f'Device: {device}  |  GPUs: {n_gpus}')
print(f'Output: {ADV_DIR}')
print(f'Lambda values: {LAMBDAS}')

---
## 2. Load & align data

In [ ]:
cv_stay_ids = np.load(OUT_DIR / 'cv_stay_ids.npy')
print(f'CV stay_ids: {len(cv_stay_ids):,}')

cohort_full = pd.read_csv(OUT_DIR / 'cohort.csv')
cohort_full = cohort_full.set_index('stay_id')
cohort_cv   = cohort_full.loc[cv_stay_ids].reset_index()

cohort_full_reset = pd.read_csv(OUT_DIR / 'cohort.csv').reset_index(drop=True)
stay_id_to_ts_idx = {sid: i for i, sid in
                     enumerate(cohort_full_reset['stay_id'].values)}

X_ts_full  = np.load(OUT_DIR / 'X_ts.npy')
M_ts_full  = np.load(OUT_DIR / 'M_ts.npy')
ts_indices = np.array([stay_id_to_ts_idx[sid] for sid in cv_stay_ids])
X_ts_cv    = X_ts_full[ts_indices]
M_ts_cv    = M_ts_full[ts_indices]

keep_mask = cohort_cv['race_group'].isin(KEEP_RACES).values
cohort    = cohort_cv[keep_mask].reset_index(drop=True)
kept_idx  = np.where(keep_mask)[0]

X_ts    = X_ts_cv[kept_idx]   # (N, 24, 5)
M_ts    = M_ts_cv[kept_idx]   # (N, 24, 5)

y        = cohort['hospital_expire_flag'].values.astype(np.int32)
demo     = cohort[['race_group', 'gender', 'age_group']].reset_index(drop=True)
race_arr = demo['race_group'].values
race_int = np.array([RACE_TO_INT[r] for r in race_arr])

print(f'Cohort: {len(cohort):,}  |  Mortality: {y.mean():.1%}')
print(cohort['race_group'].value_counts().to_string())

# Load baseline LSTM for comparison
oof_lstm_baseline = np.load(OUT_DIR / 'oof_probs_LSTM.npy')[kept_idx]
bl_black = average_precision_score(
    y[race_arr=='Black'], oof_lstm_baseline[race_arr=='Black'])
bl_white = average_precision_score(
    y[race_arr=='White'], oof_lstm_baseline[race_arr=='White'])
print(f'\nBaseline LSTM: Black={bl_black:.4f}  White={bl_white:.4f}  '
      f'Gap={bl_white-bl_black:+.4f}')

---
## 3. Model architecture

The adversarial LSTM has two components:

**LSTMEncoder:** Shared BiLSTM that produces a 256-dim representation.
This is what both the mortality classifier and the adversary see.

**MortalityClassifier:** 3-layer MLP on top of the encoder output.
Predicts mortality probability.

**RaceAdversary:** Small 2-layer MLP on top of the encoder output.
Tries to predict race group (4 classes).
The encoder is trained to FOOL this adversary.

In [ ]:
class LSTMEncoder(nn.Module):
    """Shared BiLSTM encoder — produces race-invariant representations."""
    def __init__(self, input_size=10, hidden_size=128,
                 num_layers=3, dropout=0.3):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=input_size, hidden_size=hidden_size,
            num_layers=num_layers, batch_first=True,
            dropout=dropout, bidirectional=True
        )
        self.layer_norm = nn.LayerNorm(hidden_size * 2)
        self.dropout    = nn.Dropout(dropout)
        self.repr_dim   = hidden_size * 2   # 256

    def forward(self, x):
        out, _ = self.lstm(x)
        last   = self.layer_norm(out[:, -1, :])
        return self.dropout(last)   # (batch, 256)


class MortalityClassifier(nn.Module):
    """Mortality prediction head on top of encoder representation."""
    def __init__(self, repr_dim=256, dropout=0.3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(repr_dim, 128), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(128, 64),       nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(64, 1)
        )

    def forward(self, h):
        return self.net(h).squeeze(1)


class RaceAdversary(nn.Module):
    """
    Race prediction adversary.
    Takes encoder representation and predicts race (4 classes).
    Trained to succeed; encoder trained to fool it.
    """
    def __init__(self, repr_dim=256, n_races=4, dropout=0.2):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(repr_dim, 64), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(64, 32),       nn.ReLU(),
            nn.Linear(32, n_races)
        )

    def forward(self, h):
        return self.net(h)   # logits (batch, 4)


print('Model architecture defined.')
# Parameter count
enc  = LSTMEncoder()
clf  = MortalityClassifier()
adv  = RaceAdversary()
n_enc = sum(p.numel() for p in enc.parameters())
n_clf = sum(p.numel() for p in clf.parameters())
n_adv = sum(p.numel() for p in adv.parameters())
print(f'Encoder params:    {n_enc:,}')
print(f'Classifier params: {n_clf:,}')
print(f'Adversary params:  {n_adv:,}')
del enc, clf, adv

---
## 4. Fairness metrics


In [ ]:
def compute_fairness_metrics(y_true, probs, race, threshold=THRESHOLD):
    """
    Compute fairness metrics for Black vs White comparison.
    Returns dict with all metrics.
    """
    results = {}

    for g in ['Black', 'White']:
        mask = race == g
        yg   = y_true[mask]
        pg   = probs[mask]
        pred = (pg >= threshold).astype(int)

        tp = ((pred==1) & (yg==1)).sum()
        fn = ((pred==0) & (yg==1)).sum()
        fp = ((pred==1) & (yg==0)).sum()
        tn = ((pred==0) & (yg==0)).sum()

        tpr = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        fpr = fp / (fp + tn) if (fp + tn) > 0 else 0.0
        fnr = fn / (tp + fn) if (tp + fn) > 0 else 0.0
        ber = (fpr + fnr) / 2

        results[g] = {
            'n':     int(mask.sum()),
            'auprc': average_precision_score(yg, pg),
            'auroc': roc_auc_score(yg, pg),
            'tpr':   tpr,
            'fpr':   fpr,
            'fnr':   fnr,
            'ber':   ber,
        }

    # Gaps (Black - White)
    gaps = {
        'auprc_gap':            results['White']['auprc'] - results['Black']['auprc'],
        'equal_opportunity':    abs(results['Black']['tpr'] - results['White']['tpr']),
        'equalized_odds':       max(
            abs(results['Black']['tpr'] - results['White']['tpr']),
            abs(results['Black']['fpr'] - results['White']['fpr'])
        ),
        'ber_gap':              abs(results['Black']['ber'] - results['White']['ber']),
    }

    return results, gaps


# Sanity check on baseline
bl_results, bl_gaps = compute_fairness_metrics(
    y, oof_lstm_baseline, race_arr)
print('Baseline LSTM fairness metrics:')
print(f'  AUPRC gap (White-Black):   {bl_gaps["auprc_gap"]:+.4f}')
print(f'  Equal opportunity gap:     {bl_gaps["equal_opportunity"]:.4f}')
print(f'  Equalized odds gap:        {bl_gaps["equalized_odds"]:.4f}')
print(f'  BER gap:                   {bl_gaps["ber_gap"]:.4f}')
print(f'  Black TPR: {bl_results["Black"]["tpr"]:.4f}  '
      f'White TPR: {bl_results["White"]["tpr"]:.4f}')
print(f'  Black FPR: {bl_results["Black"]["fpr"]:.4f}  '
      f'White FPR: {bl_results["White"]["fpr"]:.4f}')

---
## 5. Fold splits

In [ ]:
strat_label = (demo['race_group'].astype(str) + '_' +
               cohort['hospital_expire_flag'].astype(str)).values
skf         = StratifiedKFold(n_splits=N_FOLDS, shuffle=True,
                              random_state=RANDOM_STATE)
fold_splits  = list(skf.split(np.zeros(len(cohort)), strat_label))
print(f'Fold test sizes: {[len(t) for _,t in fold_splits]}')

---
## 6. Adversarial training loop

For each batch, three gradient steps are taken:

**Step 1: Adversary update:**
Freeze encoder + classifier. Update adversary to better predict race.
Loss: cross-entropy(adversary race predictions, true race labels)

**Step 2: Classifier update:**
Freeze encoder + adversary. Update classifier on mortality loss only.
Loss: BCE(mortality predictions, true mortality labels)

**Step 3: Encoder adversarial update:**
Freeze classifier + adversary weights.
Update encoder to minimise mortality loss AND maximise adversary loss.
Loss: mortality_loss - lambda * adversary_loss
The minus sign is key, we SUBTRACT adversary loss so the encoder
is penalised when the adversary succeeds at predicting race.

In [ ]:
def train_adversarial_fold(X_tr_in, y_tr, race_tr_int,
                            X_te_in, y_te, race_te_int,
                            lam, fold=0):
    """
    Train adversarial LSTM for one fold.
    lam: lambda — fairness constraint strength
    Returns: mortality probs (N_test,), race probs (N_test, 4)
    """
    # Class weights
    pos_weight = torch.tensor(
        [(y_tr==0).sum()/(y_tr==1).sum()],
        dtype=torch.float32).to(device)
    mort_criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    race_criterion = nn.CrossEntropyLoss()

    # Build models
    encoder    = LSTMEncoder().to(device)
    classifier = MortalityClassifier().to(device)
    adversary  = RaceAdversary().to(device)

    if n_gpus > 1:
        encoder    = nn.DataParallel(encoder)
        classifier = nn.DataParallel(classifier)
        adversary  = nn.DataParallel(adversary)

    # Separate optimisers
    opt_enc  = torch.optim.AdamW(
        encoder.parameters(), lr=LR_MAIN, weight_decay=1e-4)
    opt_clf  = torch.optim.AdamW(
        classifier.parameters(), lr=LR_MAIN, weight_decay=1e-4)
    opt_adv  = torch.optim.AdamW(
        adversary.parameters(), lr=LR_ADV, weight_decay=1e-4)

    # LR schedulers
    sched_enc = torch.optim.lr_scheduler.CosineAnnealingLR(
        opt_enc, T_max=N_EPOCHS, eta_min=1e-5)
    sched_clf = torch.optim.lr_scheduler.CosineAnnealingLR(
        opt_clf, T_max=N_EPOCHS, eta_min=1e-5)

    # Data loaders
    y_tr_t    = torch.tensor(y_tr, dtype=torch.float32)
    race_tr_t = torch.tensor(race_tr_int, dtype=torch.long)
    y_te_t    = torch.tensor(y_te, dtype=torch.float32)

    train_loader = DataLoader(
        TensorDataset(
            torch.tensor(X_tr_in), y_tr_t, race_tr_t),
        batch_size=BATCH_SIZE, shuffle=True,
        num_workers=4, pin_memory=True)
    val_loader = DataLoader(
        TensorDataset(
            torch.tensor(X_te_in), y_te_t),
        batch_size=BATCH_SIZE, shuffle=False,
        num_workers=4, pin_memory=True)

    best_auprc      = 0
    best_mort_probs = None

    for epoch in range(1, N_EPOCHS + 1):
        encoder.train(); classifier.train(); adversary.train()

        for xb, yb, rb in train_loader:
            xb = xb.to(device)
            yb = yb.to(device)
            rb = rb.to(device)

            # Step 1: Update adversary 
            with torch.no_grad():
                h = encoder(xb)
            race_logits = adversary(h.detach())
            adv_loss    = race_criterion(race_logits, rb)
            opt_adv.zero_grad()
            adv_loss.backward()
            opt_adv.step()

            # Step 2: Update classifier (mortality only)
            h           = encoder(xb)
            mort_logits = classifier(h.detach())
            mort_loss   = mort_criterion(mort_logits, yb)
            opt_clf.zero_grad()
            mort_loss.backward()
            nn.utils.clip_grad_norm_(classifier.parameters(), 1.0)
            opt_clf.step()

            # Step 3: Update encoder adversarially
            # Minimise mortality loss, MAXIMISE adversary loss
            h           = encoder(xb)
            mort_logits = classifier(h)
            race_logits = adversary(h)
            mort_loss   = mort_criterion(mort_logits, yb)
            adv_loss    = race_criterion(race_logits, rb)
            # Key: subtract lambda * adv_loss to penalise race encoding
            enc_loss    = mort_loss - lam * adv_loss
            opt_enc.zero_grad()
            enc_loss.backward()
            nn.utils.clip_grad_norm_(encoder.parameters(), 1.0)
            opt_enc.step()

        sched_enc.step()
        sched_clf.step()

        # Validation
        encoder.eval(); classifier.eval()
        mort_probs_list = []
        with torch.no_grad():
            for xb, _ in val_loader:
                h    = encoder(xb.to(device))
                mort = torch.sigmoid(classifier(h)).cpu().numpy()
                mort_probs_list.append(mort)

        mort_probs = np.concatenate(mort_probs_list)
        ap = average_precision_score(y_te, mort_probs)

        if ap > best_auprc:
            best_auprc      = ap
            best_mort_probs = mort_probs.copy()

        if epoch % 10 == 0 or epoch == 1:
            print(f'    [λ={lam} fold {fold+1}] ep {epoch:02d}  '
                  f'AUPRC={ap:.4f}{"  *" if ap==best_auprc else ""}')

    # Get final race probabilities from adversary on test set
    encoder.eval(); adversary.eval()
    race_probs_list = []
    with torch.no_grad():
        for xb, _ in val_loader:
            h    = encoder(xb.to(device))
            rp   = F.softmax(adversary(h), dim=1).cpu().numpy()
            race_probs_list.append(rp)
    race_probs = np.concatenate(race_probs_list, axis=0)

    # Save model
    base_enc = encoder.module if hasattr(encoder, 'module') else encoder
    torch.save(base_enc.state_dict(),
               ADV_DIR / f'encoder_lam{lam}_fold{fold+1}.pt')

    del encoder, classifier, adversary
    if torch.cuda.is_available(): torch.cuda.empty_cache()

    return best_mort_probs, race_probs


def normalise(X_input, X_train):
    ts_mean = X_train.mean(axis=(0,1), keepdims=True)
    ts_std  = X_train.std(axis=(0,1),  keepdims=True) + 1e-8
    return (X_input - ts_mean) / ts_std


def build_input(X_norm, M):
    return np.concatenate([X_norm, M], axis=2).astype(np.float32)


print('Adversarial training loop defined.')

---
## 7. Run adversarial training for all lambda values

In [ ]:
print(f'Starting adversarial training')
print(f'Lambda values: {LAMBDAS}')
print(f'Folds: {N_FOLDS}  |  Epochs per fold: {N_EPOCHS}')
print(f'Total training runs: {len(LAMBDAS) * N_FOLDS}')
print('=' * 65)

# Store OOF results per lambda
oof_mort   = {lam: np.zeros(len(cohort)) for lam in LAMBDAS}
# Store adversary race probs per lambda (for Black vs rest AUROC)
oof_race   = {lam: np.zeros((len(cohort), 4)) for lam in LAMBDAS}

for lam in LAMBDAS:
    print(f'\n{'='*65}')
    print(f'Lambda = {lam}')
    print(f'{'='*65}')

    for fold, (train_idx, test_idx) in enumerate(fold_splits):
        print(f'\n  Fold {fold+1}/{N_FOLDS}')

        X_tr = X_ts[train_idx]
        M_tr = M_ts[train_idx]
        X_te = X_ts[test_idx]
        M_te = M_ts[test_idx]
        y_tr = y[train_idx]
        y_te = y[test_idx]
        race_tr_int = race_int[train_idx]
        race_te_int = race_int[test_idx]

        X_tr_n  = normalise(X_tr, X_tr)
        X_te_n  = normalise(X_te, X_tr)
        X_tr_in = build_input(X_tr_n, M_tr)
        X_te_in = build_input(X_te_n, M_te)

        mort_probs, race_probs = train_adversarial_fold(
            X_tr_in, y_tr, race_tr_int,
            X_te_in, y_te, race_te_int,
            lam=lam, fold=fold
        )

        oof_mort[lam][test_idx]     = mort_probs
        oof_race[lam][test_idx, :]  = race_probs

        # Quick fold summary
        race_te = race_arr[test_idx]
        for g in ['Black', 'White']:
            mask = race_te == g
            if mask.sum() > 0 and y_te[mask].sum() > 0:
                ap = average_precision_score(y_te[mask], mort_probs[mask])
                print(f'    {g}: AUPRC={ap:.4f}')

    # Save OOF probs
    np.save(ADV_DIR / f'oof_mort_lam{lam}.npy', oof_mort[lam])
    np.save(ADV_DIR / f'oof_race_lam{lam}.npy', oof_race[lam])
    print(f'\nLambda {lam} saved.')

print('\nAll lambda values complete!')

---
## 8. Results

In [ ]:
print('Computing fairness metrics for all lambda values...')
print('=' * 75)

all_results = {}
summary_rows = []

# Black vs rest: adversary probability for Black class
y_black_vs_rest = (race_arr == 'Black').astype(int)

for lam in LAMBDAS:
    probs    = oof_mort[lam]
    rprobs   = oof_race[lam]

    res, gaps = compute_fairness_metrics(y, probs, race_arr)

    # Adversary AUROC
    black_idx = RACE_TO_INT['Black']   # = 1
    adv_auroc = roc_auc_score(
        y_black_vs_rest, rprobs[:, black_idx])

    overall_auprc = average_precision_score(y, probs)

    all_results[lam] = {
        'overall_auprc':    overall_auprc,
        'black_auprc':      res['Black']['auprc'],
        'white_auprc':      res['White']['auprc'],
        'black_tpr':        res['Black']['tpr'],
        'white_tpr':        res['White']['tpr'],
        'black_fpr':        res['Black']['fpr'],
        'white_fpr':        res['White']['fpr'],
        'black_ber':        res['Black']['ber'],
        'white_ber':        res['White']['ber'],
        'auprc_gap':        gaps['auprc_gap'],
        'equal_opportunity':gaps['equal_opportunity'],
        'equalized_odds':   gaps['equalized_odds'],
        'ber_gap':          gaps['ber_gap'],
        'adv_auroc':        adv_auroc,
    }

    summary_rows.append({'lambda': lam, **all_results[lam]})

    print(f'\nLambda = {lam}:')
    print(f'  Overall AUPRC:       {overall_auprc:.4f}')
    print(f'  Black AUPRC:         {res["Black"]["auprc"]:.4f}')
    print(f'  White AUPRC:         {res["White"]["auprc"]:.4f}')
    print(f'  AUPRC gap:           {gaps["auprc_gap"]:+.4f}')
    print(f'  Equal opportunity:   {gaps["equal_opportunity"]:.4f}  '
          f'(Black TPR={res["Black"]["tpr"]:.3f}  White TPR={res["White"]["tpr"]:.3f})')
    print(f'  Equalized odds:      {gaps["equalized_odds"]:.4f}')
    print(f'  BER gap:             {gaps["ber_gap"]:.4f}')
    print(f'  Adversary AUROC:     {adv_auroc:.4f}  '
          f'(0.500 = adversary cannot predict race)')

summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv(ADV_DIR / 'adversarial_summary.csv', index=False)
print('\nSaved adversarial_summary.csv')

---
## 9. Figures

In [ ]:
# Figure 1: Fairness-performance tradeoff 
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

lambdas_plot = LAMBDAS

# Overall AUPRC
ax = axes[0, 0]
overall_auprcs = [all_results[l]['overall_auprc'] for l in lambdas_plot]
ax.plot(lambdas_plot, overall_auprcs, 'o-', color='#2c3e50',
        linewidth=2.5, markersize=8)
ax.axhline(all_results[0.0]['overall_auprc'], color='gray',
           linestyle=':', alpha=0.5, label='Lambda=0 baseline')
ax.set_xlabel('Lambda (fairness constraint strength)')
ax.set_ylabel('Overall AUPRC')
ax.set_title('Overall AUPRC vs Lambda\n(performance cost of fairness)')
ax.set_xticks(lambdas_plot)
for x, y_val in zip(lambdas_plot, overall_auprcs):
    ax.text(x, y_val+0.001, f'{y_val:.4f}', ha='center', fontsize=9)

# Black vs White AUPRC
ax = axes[0, 1]
black_auprcs = [all_results[l]['black_auprc'] for l in lambdas_plot]
white_auprcs = [all_results[l]['white_auprc'] for l in lambdas_plot]
ax.plot(lambdas_plot, black_auprcs, 'o-', color='#D65F5F',
        linewidth=2.5, markersize=8, label='Black')
ax.plot(lambdas_plot, white_auprcs, 's-', color='#4878CF',
        linewidth=2.5, markersize=8, label='White')
ax.fill_between(lambdas_plot, black_auprcs, white_auprcs,
                alpha=0.1, color='gray', label='Gap')
ax.set_xlabel('Lambda')
ax.set_ylabel('AUPRC')
ax.set_title('Black vs White AUPRC\nas lambda increases')
ax.legend(fontsize=9)
ax.set_xticks(lambdas_plot)

# Fairness metrics gap
ax = axes[1, 0]
eq_opp  = [all_results[l]['equal_opportunity'] for l in lambdas_plot]
eq_odds = [all_results[l]['equalized_odds']    for l in lambdas_plot]
ber_gap = [all_results[l]['ber_gap']           for l in lambdas_plot]
ax.plot(lambdas_plot, eq_opp,  'o-', color='#27ae60',
        linewidth=2.5, markersize=8, label='Equal opportunity gap')
ax.plot(lambdas_plot, eq_odds, 's-', color='#8e44ad',
        linewidth=2.5, markersize=8, label='Equalized odds gap')
ax.plot(lambdas_plot, ber_gap, '^-', color='#e67e22',
        linewidth=2.5, markersize=8, label='BER gap')
ax.axhline(0, color='black', linestyle='--', linewidth=0.8,
           label='Perfect fairness')
ax.set_xlabel('Lambda')
ax.set_ylabel('Fairness metric gap (Black vs White)')
ax.set_title('Fairness metrics vs Lambda\n(lower = more equitable)')
ax.legend(fontsize=8)
ax.set_xticks(lambdas_plot)

# Adversary AUROC
ax = axes[1, 1]
adv_aurocs = [all_results[l]['adv_auroc'] for l in lambdas_plot]
ax.plot(lambdas_plot, adv_aurocs, 'o-', color='#c0392b',
        linewidth=2.5, markersize=8, label='Adversary AUROC')
ax.axhline(0.5, color='gray', linestyle='--', linewidth=1.5,
           label='Random baseline (0.500)')
ax.set_xlabel('Lambda')
ax.set_ylabel('Adversary AUROC (Black vs rest)')
ax.set_title('Can adversary still predict race?\n'
             '0.5 = completely blind to race')
ax.legend(fontsize=9)
ax.set_xticks(lambdas_plot)
for x, y_val in zip(lambdas_plot, adv_aurocs):
    ax.text(x, y_val+0.002, f'{y_val:.3f}', ha='center', fontsize=9)

plt.suptitle('Adversarial debiasing — fairness-performance tradeoff\n'
             'LSTM penalised for encoding race in its representations',
             fontsize=13)
plt.tight_layout()
plt.savefig(ADV_DIR / 'fig_adversarial_tradeoff.png',
            dpi=150, bbox_inches='tight')
plt.show()
print('Saved fig_adversarial_tradeoff.png')

In [ ]:
# Figure 2: AUPRC by race with all lambda values
fig, ax = plt.subplots(figsize=(12, 5))

x     = np.arange(len(KEEP_RACES))
width = 0.18

for offset, lam in enumerate(LAMBDAS):
    vals = []
    for g in KEEP_RACES:
        mask = race_arr == g
        vals.append(average_precision_score(
            y[mask], oof_mort[lam][mask]))
    ax.bar(x + offset*width, vals, width,
           label=f'λ={lam}',
           color=LAMBDA_COLORS[lam],
           edgecolor='white', alpha=0.85)

ax.set_xticks(x + width*1.5)
ax.set_xticklabels(KEEP_RACES)
ax.set_ylabel('AUPRC')
ax.set_title('AUPRC by race group — all lambda values\n'
             'Does adversarial training reduce the gap?')
ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig(ADV_DIR / 'fig_adversarial_auprc_by_race.png',
            dpi=150, bbox_inches='tight')
plt.show()
print('Saved fig_adversarial_auprc_by_race.png')

In [ ]:
# Figure 3: Fairness metrics summary bar chart
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

metric_keys = ['equal_opportunity', 'equalized_odds', 'ber_gap']
metric_labels = ['Equal Opportunity\ngap (TPR)', 'Equalized Odds\ngap', 'BER gap']
metric_colors = ['#27ae60', '#8e44ad', '#e67e22']

for ax, key, label, color in zip(axes, metric_keys, metric_labels, metric_colors):
    vals = [all_results[l][key] for l in LAMBDAS]
    bars = ax.bar([str(l) for l in LAMBDAS], vals,
                  color=color, edgecolor='white', alpha=0.85)
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x()+bar.get_width()/2, val+0.002,
                f'{val:.4f}', ha='center', fontsize=9, fontweight='bold')
    ax.set_xlabel('Lambda')
    ax.set_ylabel('Gap (Black vs White)')
    ax.set_title(f'{label}\nLower = more equitable')
    ax.axhline(0, color='black', linestyle='--', linewidth=0.8)

plt.suptitle('Fairness metrics across lambda values\n'
             'Black vs White gaps', fontsize=13)
plt.tight_layout()
plt.savefig(ADV_DIR / 'fig_adversarial_fairness_metrics.png',
            dpi=150, bbox_inches='tight')
plt.show()
print('Saved fig_adversarial_fairness_metrics.png')

In [ ]:
# Figure 4: PR curves for Black patients with all lambda values
fig, ax = plt.subplots(figsize=(8, 6))

mask_b = race_arr == 'Black'
yb     = y[mask_b]

ax.axhline(yb.mean(), color='gray', linestyle=':', alpha=0.6,
           label=f'Prevalence ({yb.mean():.2f})')

for lam in LAMBDAS:
    pb   = oof_mort[lam][mask_b]
    ap   = average_precision_score(yb, pb)
    prec, rec, _ = precision_recall_curve(yb, pb)
    ax.plot(rec, prec, color=LAMBDA_COLORS[lam], linewidth=2.5,
            label=f'λ={lam} (AUPRC={ap:.3f})')

ax.set_xlabel('Recall')
ax.set_ylabel('Precision')
ax.set_title('PR curves — Black patients only\nAll lambda values')
ax.legend(fontsize=10)
ax.set_xlim(0,1); ax.set_ylim(0,1)
plt.tight_layout()
plt.savefig(ADV_DIR / 'fig_adversarial_pr_black.png',
            dpi=150, bbox_inches='tight')
plt.show()
print('Saved fig_adversarial_pr_black.png')

---
## 10. Summary

In [ ]:
print('=' * 75)
print('ADVERSARIAL DEBIASING SUMMARY')
print('=' * 75)

print(f'\n{"Lambda":<8} {"Overall":>8} {"Black":>8} {"White":>8} '
      f'{"Gap":>8} {"Eq.Opp":>8} {"Eq.Odds":>8} {"BER":>8} {"AdvAUROC":>10}')
print('-' * 80)

for lam in LAMBDAS:
    r = all_results[lam]
    print(f'{lam:<8} {r["overall_auprc"]:>8.4f} {r["black_auprc"]:>8.4f} '
          f'{r["white_auprc"]:>8.4f} {r["auprc_gap"]:>+8.4f} '
          f'{r["equal_opportunity"]:>8.4f} {r["equalized_odds"]:>8.4f} '
          f'{r["ber_gap"]:>8.4f} {r["adv_auroc"]:>10.4f}')

print('\n── Interpretation ──')
best_lam = min(LAMBDAS[1:],
               key=lambda l: all_results[l]['equal_opportunity'])
print(f'Best fairness (equal opportunity): lambda={best_lam}')
print(f'  Equal opportunity gap reduced from '
      f'{all_results[0.0]["equal_opportunity"]:.4f} to '
      f'{all_results[best_lam]["equal_opportunity"]:.4f}')
print(f'  Overall AUPRC change: '
      f'{all_results[best_lam]["overall_auprc"]-all_results[0.0]["overall_auprc"]:+.4f}')
print(f'  Adversary AUROC: {all_results[best_lam]["adv_auroc"]:.4f} '
      f'(baseline: {all_results[0.0]["adv_auroc"]:.4f})')

print(f'\nOutputs saved to: {ADV_DIR}')
for f in sorted(ADV_DIR.glob('*.png')):
    print(f'  {f.name}')
for f in sorted(ADV_DIR.glob('*.csv')):
    print(f'  {f.name}')